# SIRIUS Phase 2 — QLoRA train → prune → quantize
Run cells in order. Cell 1 installs deps, cell 2 uploads the four input files, cell 3 trains, cell 4 checks the PPL gate, cell 6 prunes/quantizes, cell 7 downloads artifacts.

In [ ]:
# CELL 1 — install (run first)
!pip install -q -U transformers==4.46.0 peft==0.13.0 bitsandbytes==0.44.1 datasets==3.1.0 accelerate==1.1.1 trl==0.12.0
!nvidia-smi

In [ ]:
# CELL 2 — upload inputs: sirius_train.jsonl, sirius_dev.jsonl,
# 07_train_colab_fixed.py, 08_prune_quantize_colab_fixed.py
from google.colab import files
import os
uploaded = files.upload()
print('uploaded:', list(uploaded.keys()))
print('/content:', [f for f in os.listdir('/content') if f.endswith(('.py', '.jsonl'))])

In [ ]:
# CELL 3 — train (primary config: all 7 modules, r=16)
!python /content/07_train_colab_fixed.py --skip-install-check

In [ ]:
# CELL 4 — check gate and package merged model for download
import json, os, shutil
log = json.load(open('/content/sirius_train_log.json'))
print('PPL0:', log['ppl0_reference'], ' PPL after:', log['ppl_after_training'])
print('PPL delta %:', log['ppl_delta_pct'], ' gate passed:', log['ppl_gate_passed'])
print('next:', log['next_step_if_gate_fails'])
if not log['ppl_gate_passed']:
    print('>>> DE-ESCALATE: rerun CELL 3 with --lora-config-idx 1, then 2')
else:
    shutil.make_archive('/content/sirius_merged_pass1', 'zip', '/content/sirius_merged_pass1')
    print('merged zipped -> /content/sirius_merged_pass1.zip')

In [ ]:
# CELL 5 — download train log + merged model
from google.colab import files
files.download('/content/sirius_train_log.json')
if os.path.isfile('/content/sirius_merged_pass1.zip'):
    files.download('/content/sirius_merged_pass1.zip')

In [ ]:
# CELL 6 — ShortGPT prune + quantize (uses PPL0 from the train log)
!python /content/08_prune_quantize_colab_fixed.py --merged-dir /content/sirius_merged_pass1 --adapter-log /content/sirius_train_log.json

In [ ]:
# CELL 7 — download GGUFs + Modelfile + prune log
from google.colab import files
import os
for f in ['/content/sirius_final_Q6_K.gguf', '/content/sirius_final_Q5_K_M.gguf',
          '/content/Modelfile.sirius', '/content/sirius_prune_quantize_log.json']:
    if os.path.isfile(f):
        print('downloading', f)
        files.download(f)
    else:
        print('MISSING', f)